# 04. 주입 (recipe_db_lab)
03_vecs -> ChromaDB recipe_db_lab (서빙용 recipe_db 보호). 실주입은 scripts/ingest_recipes.py

In [ ]:
import sys,os,pickle
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
from dotenv import load_dotenv; load_dotenv()
from app.rag.embedder import CohereEmbedder
from app.rag.indexer import get_chroma_client,_sanitize_metadata
d=pickle.loads((B/'data'/'lab'/'03_vecs.pkl').read_bytes()); recs,vecs=d['recs'],d['vecs']
sig=CohereEmbedder().signature
c=get_chroma_client()
try: c.delete_collection('recipe_db_lab')
except Exception: pass
col=c.create_collection('recipe_db_lab',metadata={'hnsw:space':'cosine','embed_sig':sig})
col.add(ids=[r['id'] for r in recs],documents=[r['embed_text'] for r in recs],embeddings=vecs,metadatas=[_sanitize_metadata(r) for r in recs])
print('recipe_db_lab indexed:',col.count(),'sig:',sig)